# Candidate Generation: Content Two-Tower MLP with Popularity Features

Personalised content-and-popularity candidate generator that scores a request against every eligible article using two independent MLPs. The user tower reads pooled intrinsic content; the item tower reads intrinsic content plus causal prior-hour exposure and click counts, with no user or article IDs.

Protocol:

- Build positives from session clicks in the full `labels_train` pool and negatives from exposed non-clicks in the same slate.
- Redraw up to four exposed negatives per positive from the same slate on every epoch.
- Build each request feature from `full_history_rows` only: mean-pooled `text64` and `entity100`, L2-normalised per block, plus two availability flags.
- Build each item feature from intrinsic content plus `log1p` of the previous completed hour's exposures and clicks for that article.
- Train BPR with AdamW, selecting the epoch by NDCG on a chronological 90% cut of the training period.
- Tune additive prior-hour exposure and click weights on the tuning slates for serving only, then freeze them.
- Refit on the full training period for the selected number of epochs.
- Score each request over articles with `available_from <= timestamp`, excluding the request's `full_history_rows`.
- Add the frozen popularity contribution to the two-tower score at serving; this term never enters the training loss.
- Break ties by the previous completed hour's click popularity descending, then ascending `news_code`; backfill short lists from popularity.
- Write the standard `impression_uid, cutoff, news_code` table and evaluate it with `cg_evaluation.py`.

The user tower consumes only intrinsic history content. The item tower and the serving score consume strictly causal prior-hour popularity; no same-hour or target-period feedback is used. `FEATURE_DIM = 166`, `ITEM_FEATURE_DIM = 168`, `HIDDEN_DIM = 256`, `EMBED_DIM = 256` and the 500-candidate output are fixed in this notebook.


## 0. Configure

**This is the only cell you need to edit.** Set `REUSE_ARTIFACTS = False` to retrain the model
and rebuild every artifact. Candidate lists are always regenerated for `TARGET_SPLIT`, so a new
validation or test split is served normally.


In [ ]:
# ---- Where to run ----------------------------------------------------------------------------
# "colab" = Google Colab with Google Drive, "local" = this computer.
ENVIRONMENT = "colab"  # "local" or "colab"

# ---- Paths -------------------------------------------------------------------------------------
# Data path: the prepared data. It must contain
# 1_Data_Preparation/processed/, 3_Representations/artifacts/ and
# 4_Feature_Engineering/artifacts/prequential-all/.
# On Colab, the zipped data in <submission>/2.Feature_Engineering is unpacked into the data
# path on the first run (a few minutes); later runs reuse it.
COLAB_DATA_PATH = "/content/drive/MyDrive/submission/2.Feature_Engineering"
LOCAL_DATA_PATH = None  # used locally; None = the project folder

# ---- Experiment ------------------------------------------------------------------------------
# Split to evaluate: "test" for the final results, or "validation".
TARGET_SPLIT = "validation"

# True: load the saved model checkpoint and skip training. Serving still runs, so candidate
# lists are always regenerated for TARGET_SPLIT.
# False: retrain the model from scratch.
# Baselines without a model reuse any cached intermediate state, but their candidate lists are
# still regenerated.
REUSE_ARTIFACTS = True

# ---- Reused artifacts when REUSE_ARTIFACTS=True -----------------------------------------------
# Loaded checkpoint, so training and the serving-weight search are skipped:
#   artifacts/two_tower_content_popularity_bpr/model.pt
# Candidate lists are always regenerated.


## 1. Setup


In [ ]:
from __future__ import annotations

import json
import os
import sys
import time
from pathlib import Path

import numpy as np
import polars as pl
import torch
import torch.nn.functional as F
from torch import nn


if ENVIRONMENT == "colab":
    from google.colab import drive

    drive.mount("/content/drive")
    CODE_ROOT = Path("/content/drive/MyDrive/submission/4.Code")
else:
    CODE_ROOT = next(
        p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_all.py").is_file()
    )
ROOT = CODE_ROOT
DATA_PATH = COLAB_DATA_PATH if ENVIRONMENT == "colab" else LOCAL_DATA_PATH
DATA_ROOT = (
    Path(
        DATA_PATH
        or os.environ.get("NEWSREC_DATA_ROOT")
        or os.environ.get("NEWSREC_ROOT")
        or CODE_ROOT
    )
    .expanduser()
    .resolve()
)
REQUIRED_DATA = [
    "1_Data_Preparation/processed",
    "3_Representations/artifacts",
    "4_Feature_Engineering/artifacts/prequential-all",
]
SUBMISSION_ROOT = CODE_ROOT.parent if ENVIRONMENT == "colab" else None
BUNDLE_DIR = SUBMISSION_ROOT / "2.Feature_Engineering" if SUBMISSION_ROOT else None
if BUNDLE_DIR is not None and BUNDLE_DIR.is_dir():
    import zipfile

    for archive in sorted(BUNDLE_DIR.glob("*.zip")):
        marker = DATA_ROOT / f".unpacked_{archive.stem}"
        if not marker.exists():
            print(f"Unpacking {archive.name} into {DATA_ROOT} (first run only) ...")
            with zipfile.ZipFile(archive) as bundle:
                bundle.extractall(DATA_ROOT)
            marker.touch()

missing_data = [
    str(DATA_ROOT / sub) for sub in REQUIRED_DATA if not (DATA_ROOT / sub).exists()
]
if missing_data:
    raise FileNotFoundError(
        f"{ENVIRONMENT.upper()}_DATA_PATH is missing these folders:\n"
        + "\n".join(missing_data)
    )
REPLAY_ROOT = DATA_ROOT
CG_DIR = CODE_ROOT / "5_Models" / "5_01_Candidate_Generation"
sys.path.insert(0, str(CG_DIR))
sys.path.insert(0, str(CODE_ROOT / "src"))

from cf_models import resolve_device
from cg_result_writer import CandidateWriter
from neural_helpers import (
    EarlyStopper,
    EpochNegativeSampler,
    chronological_cutoff,
    chronological_mask,
    max_clicks_per_impression,
    mean_slate_metrics,
    tuning_slates,
)

from newsrec.common import Settings

PROCESSED = DATA_ROOT / "1_Data_Preparation" / "processed"
FEATURES = DATA_ROOT / "4_Feature_Engineering" / "artifacts" / "prequential-all"
REPRESENTATIONS = DATA_ROOT / "3_Representations" / "artifacts"
ARTIFACTS = (
    CODE_ROOT.parent / "3.Model_Artifacts" / "CG"
    if ENVIRONMENT == "colab"
    else DATA_ROOT / "5_Models" / "5_01_Candidate_Generation" / "artifacts"
)
RESULTS = DATA_ROOT / "results"

FIT_SPLIT = "train"
BASE_MODEL_NAME = "content_two_tower_popularity_mlp"
RUN_LOSSES = ("bpr",)
LOSS_CONFIGS = {}
for loss in RUN_LOSSES:
    artifact_dir = ARTIFACTS / f"two_tower_content_popularity_{loss}"
    artifact_dir.mkdir(parents=True, exist_ok=True)
    LOSS_CONFIGS[loss] = {
        "model_name": f"{BASE_MODEL_NAME}_{loss}",
        "artifact_dir": artifact_dir,
        "recommendations_path": artifact_dir
        / f"recommendations_{TARGET_SPLIT}_top500.parquet",
        "result_path": RESULTS
        / f"candidate_generation_{BASE_MODEL_NAME}_{loss}_{TARGET_SPLIT}.csv",
    }
PROFILE_DIR = ARTIFACTS / "two_tower_content_popularity_bpr"
PROFILE_DIR.mkdir(parents=True, exist_ok=True)

M_VALUES = (50, 100, 200, 500)
CANDIDATE_CAP = max(M_VALUES)
TEXT_DIM = 64
ENTITY_DIM = 100
FEATURE_DIM = TEXT_DIM + ENTITY_DIM + 2
POPULARITY_FEATURES = ("prior_hour_exposures", "prior_hour_clicks")
POP_DIM = len(POPULARITY_FEATURES)
ITEM_FEATURE_DIM = FEATURE_DIM + POP_DIM
HIDDEN_DIM = 256
EMBED_DIM = 256
DROPOUT = 0.1
NUM_EPOCHS = 50
BATCH_SIZE = 16_384
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
SEED = 4222
PATIENCE = 3
TUNE_QUANTILE = 0.9
TUNE_CANDIDATES = PROCESSED / "candidates_train.parquet"
DEVICE = resolve_device("auto")
SERVING_WEIGHT_GRID = (0.0, 0.25, 0.5, 1.0, 2.0)

HOUR_NS = 3_600 * 10**9
POPULARITY_SPLITS = ("train", "validation")

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)
print(f"Item input dim: {ITEM_FEATURE_DIM}")

In [ ]:
def _checkpoint_matches(path, expected):
    if not path.exists():
        return False
    try:
        saved = torch.load(path, map_location="cpu", weights_only=False).get("config", {})
    except Exception:
        return False
    return all(saved.get(key) == value for key, value in expected.items())

TRAINING_REQUIRED = not all(
    _checkpoint_matches(
        LOSS_CONFIGS[objective]["artifact_dir"] / "model.pt",
        {
            "user_input_dim": FEATURE_DIM,
            "item_input_dim": ITEM_FEATURE_DIM,
            "hidden_dim": HIDDEN_DIM,
            "embed_dim": EMBED_DIM,
            "dropout": DROPOUT,
            "popularity_features": list(POPULARITY_FEATURES),
            "objective": objective,
        },
    )
    for objective in RUN_LOSSES
)


## 2. Data ingestion

Each request profile is built causally from its `full_history_rows`: non-zero `text64` and `entity100` vectors are mean-pooled, L2-normalised per block, and two flags record whether each block was present, giving a 166-dimensional input cached per split. Article inputs use the same content blocks and flags, indexed by `article_row` with row 0 reserved for padding. An hourly engagement matrix is built from training and validation slates with both exposure and click counts and read one completed hour behind the request, so ordering is strictly causal. The prior-hour exposure and click counts are appended to the item content vector as `log1p` features, giving a 168-dimensional item input. The same causal popularity values are used for tie-breaking, the empty-history fallback and the serving-only additive score. Positives are session clicks from the full `labels_train` pool; each epoch redraws up to four exposed non-clicks from the same slate, restricted to the chronological fit window for tuning and applied to all rows for the refit. Tuning slates are sampled deterministically from `candidates_train`.


In [42]:
def pool_profiles(history_rows, matrix, block=2_000):
    matrix = np.asarray(matrix)
    n = len(history_rows)
    out = np.zeros((n, matrix.shape[1]), dtype=np.float32)
    for start in range(0, n, block):
        chunk = history_rows[start : start + block]
        lengths = np.fromiter(
            (len(rows) for rows in chunk), dtype=np.int64, count=len(chunk)
        )
        if lengths.sum() == 0:
            continue
        flat = np.concatenate(
            [np.asarray(rows, dtype=np.int64) for rows in chunk if len(rows)]
        )
        vectors = np.asarray(matrix[flat], dtype=np.float32)
        valid = np.linalg.norm(vectors, axis=1) > 1e-12
        if not valid.any():
            continue
        groups = np.repeat(np.arange(len(chunk), dtype=np.int64), lengths)[valid]
        vectors = vectors[valid]
        starts = np.r_[0, np.flatnonzero(groups[1:] != groups[:-1]) + 1]
        pooled = (
            np.add.reduceat(vectors, starts, axis=0)
            / np.diff(np.r_[starts, len(groups)])[:, None]
        )
        norms = np.linalg.norm(pooled, axis=1)
        pooled /= np.where(norms > 0, norms, 1)[:, None]
        out[start : start + len(chunk)][groups[starts]] = pooled
    return out

In [43]:
def profile_features(history_rows):
    text_profile = pool_profiles(history_rows, TEXT)
    entity_profile = pool_profiles(history_rows, ENTITY)
    out = np.zeros((len(history_rows), FEATURE_DIM), dtype=np.float32)
    out[:, :TEXT_DIM] = text_profile
    out[:, TEXT_DIM : TEXT_DIM + ENTITY_DIM] = entity_profile
    out[:, -2] = np.linalg.norm(text_profile, axis=1) > 0
    out[:, -1] = np.linalg.norm(entity_profile, axis=1) > 0
    return out

In [44]:
def content_features(text, entity):
    out = np.zeros((text.shape[0], FEATURE_DIM), dtype=np.float32)
    out[:, :TEXT_DIM] = text
    out[:, TEXT_DIM : TEXT_DIM + ENTITY_DIM] = entity
    out[:, -2] = np.linalg.norm(text, axis=1) > 0
    out[:, -1] = np.linalg.norm(entity, axis=1) > 0
    return out

In [45]:
def load_profiles(split):
    parts = sorted((FEATURES / f"request_profiles_{split}").glob("*.parquet"))
    return pl.read_parquet(
        parts, columns=["impression_uid", "timestamp", "full_history_rows"]
    ).sort("timestamp", maintain_order=True)

In [46]:
def profile_cache_paths(split):
    return (
        PROFILE_DIR / f"user_features_{split}.npy",
        PROFILE_DIR / f"profile_uids_{split}.npy",
        PROFILE_DIR / f"profile_index_{split}.parquet",
    )

In [47]:
def load_profile_cache(split):
    feature_path, uid_path, index_path = profile_cache_paths(split)
    if feature_path.exists() and uid_path.exists() and index_path.exists():
        return np.load(feature_path, mmap_mode="r"), np.load(uid_path)

    frame = load_profiles(split)
    n = frame.height
    features = np.lib.format.open_memmap(
        feature_path, mode="w+", dtype=np.float32, shape=(n, FEATURE_DIM)
    )
    features[:] = profile_features(frame["full_history_rows"].to_list())
    features.flush()
    uids = frame["impression_uid"].to_numpy()
    np.save(uid_path, uids)
    pl.DataFrame(
        {"impression_uid": uids, "profile_row": np.arange(n, dtype=np.int32)}
    ).write_parquet(index_path)
    print(f"Built profile cache for {split}: {n:,} requests")
    return np.load(feature_path, mmap_mode="r"), uids

In [48]:
TEXT = np.load(REPRESENTATIONS / "text64.npy", mmap_mode="r")
ENTITY = np.load(REPRESENTATIONS / "article_entity100.npy", mmap_mode="r")

TRAIN_USER_FEATURES, _ = load_profile_cache(FIT_SPLIT)
PROFILE_INDEX_TRAIN = pl.read_parquet(profile_cache_paths(FIT_SPLIT)[2])

print(f"Train user feature shape: {TRAIN_USER_FEATURES.shape}")

Train user feature shape: (95071, 166)


In [49]:
article_index = pl.read_parquet(
    REPRESENTATIONS / "article_index.parquet", columns=["article_row", "news_code"]
).sort("article_row")
n_items = article_index.height + 1

availability = pl.read_parquet(
    REPRESENTATIONS / "article_availability.parquet",
    columns=["news_code", "available_from"],
)
catalogue = article_index.join(availability, on="news_code", how="left")
ordered = catalogue.sort(["available_from", "news_code"])
ARTICLE_ROWS_SORTED = ordered["article_row"].to_numpy().astype(np.int64)
NEWS_SORTED = ordered["news_code"].to_numpy().astype(np.int64)
AVAILABLE_SORTED = ordered["available_from"].to_numpy().astype("datetime64[ns]")

CONTENT_INPUTS = content_features(TEXT, ENTITY)
CONTENT_INPUTS_SORTED = CONTENT_INPUTS[ARTICLE_ROWS_SORTED]
POS_IN_SORTED = np.full(n_items, -1, dtype=np.int64)
POS_IN_SORTED[ARTICLE_ROWS_SORTED] = np.arange(len(ARTICLE_ROWS_SORTED))

print(f"Catalogue articles: {article_index.height:,}")

Catalogue articles: 51,282


In [50]:
index_codes = article_index["news_code"].to_numpy().astype(np.int64)
row_of_code = np.zeros(int(index_codes.max()) + 1, dtype=np.int64)
row_of_code[index_codes] = article_index["article_row"].to_numpy().astype(np.int64)

hourly = (
    pl.concat(
        [
            pl.scan_parquet(PROCESSED / f"candidates_{split}.parquet").select(
                "news_code",
                "clicked",
                (pl.col("timestamp").dt.epoch("ns") // HOUR_NS)
                .cast(pl.Int64)
                .alias("hour"),
            )
            for split in POPULARITY_SPLITS
        ]
    )
    .group_by("news_code", "hour")
    .agg(
        exp=pl.len(),
        clk=pl.col("clicked").cast(pl.Int64).sum(),
    )
    .collect()
)

source_rows = row_of_code[hourly["news_code"].to_numpy().astype(np.int64)]
MIN_HOUR = int(hourly["hour"].min())
MAX_HOUR = int(hourly["hour"].max())
N_HOURS = MAX_HOUR - MIN_HOUR + 1
EXPOSURES = np.zeros((N_HOURS, n_items), dtype=np.int32)
CLICKS = np.zeros((N_HOURS, n_items), dtype=np.int32)
hour_positions = hourly["hour"].to_numpy().astype(np.int64) - MIN_HOUR
EXPOSURES[hour_positions, source_rows] = hourly["exp"].to_numpy().astype(np.int32)
CLICKS[hour_positions, source_rows] = hourly["clk"].to_numpy().astype(np.int32)


def hour_index_of(timestamps_ns):
    return np.asarray(timestamps_ns, dtype=np.int64) // HOUR_NS - MIN_HOUR


def prior_hour_counts(hour_index, item_rows):
    item_rows = np.asarray(item_rows, dtype=np.int64)
    hours = np.asarray(hour_index, dtype=np.int64)
    if hours.ndim == 0:
        hours = np.full(item_rows.shape, int(hours), dtype=np.int64)
    elif hours.shape != item_rows.shape:
        if hours.ndim == 1 and hours.shape[0] == item_rows.shape[0]:
            hours = np.broadcast_to(hours[:, None], item_rows.shape)
        else:
            hours = np.broadcast_to(hours, item_rows.shape)
    out = np.zeros((2, *item_rows.shape), dtype=np.float32)
    valid = hours >= 1
    if valid.any():
        rows = item_rows[valid]
        prior = hours[valid] - 1
        out[0][valid] = np.log1p(EXPOSURES[prior, rows].astype(np.float32))
        out[1][valid] = np.log1p(CLICKS[prior, rows].astype(np.float32))
    return out[0], out[1]


def item_features(item_rows, hours):
    item_rows = np.asarray(item_rows, dtype=np.int64)
    out = np.zeros((*item_rows.shape, ITEM_FEATURE_DIM), dtype=np.float32)
    out[..., :FEATURE_DIM] = CONTENT_INPUTS[item_rows]
    exposures, clicks = prior_hour_counts(hours, item_rows)
    out[..., FEATURE_DIM] = exposures
    out[..., FEATURE_DIM + 1] = clicks
    return out


print(f"Rolling popularity hours: {N_HOURS:,}")
print(f"Articles exposed at least once: {int((EXPOSURES.sum(0) > 0).sum()):,}")
print(f"Articles clicked at least once: {int((CLICKS.sum(0) > 0).sum()):,}")
print(f"Item input dim: {ITEM_FEATURE_DIM}")

Rolling popularity hours: 120
Articles exposed at least once: 16,978
Articles clicked at least once: 6,398
Item input dim: 168


In [51]:
if TRAINING_REQUIRED:
    run_manifest = json.loads((FEATURES / "run_manifest.json").read_text())
    if run_manifest.get("training_candidate_pool") != "complete":
        raise ValueError(
            "Per-epoch sampling needs the complete training pool. "
            "Rebuild stage 4 with the default training_negative_sampling='epoch'."
        )
    CFG = Settings(**run_manifest["config"])

    TRAINING_ROWS = (
        pl.read_parquet(sorted((FEATURES / "labels_train").glob("*.parquet")))
        .select("impression_uid", "news_code", "clicked")
        .to_pandas()
    )
    if (
        TRAINING_ROWS["clicked"].isna().any()
        or not TRAINING_ROWS["clicked"].isin([0, 1]).all()
    ):
        raise ValueError("Training rows must be uniquely binary-labelled")

    train_profile_index = pl.read_parquet(
        sorted((FEATURES / f"request_profiles_{FIT_SPLIT}").glob("*.parquet")),
        columns=["impression_uid", "timestamp"],
    )
    profile_lookup = PROFILE_INDEX_TRAIN.join(
        train_profile_index, on="impression_uid", how="left"
    ).with_columns(
        ((pl.col("timestamp").dt.epoch("ns") // HOUR_NS).cast(pl.Int64) - MIN_HOUR).alias(
            "hour"
        )
    )
    if profile_lookup["hour"].null_count():
        raise ValueError("Training profiles are missing timestamps")

    positive_uids = np.unique(
        TRAINING_ROWS.loc[TRAINING_ROWS["clicked"].eq(1), "impression_uid"].to_numpy()
    )
    positive_profiles = profile_lookup.filter(pl.col("impression_uid").is_in(positive_uids))
    cutoff_ns = chronological_cutoff(
        positive_profiles["timestamp"].to_numpy(), TUNE_QUANTILE
    )
    fit_profiles = chronological_mask(profile_lookup["timestamp"].to_numpy(), cutoff_ns)
    fit_uids = set(
        profile_lookup.filter(pl.Series("fit", fit_profiles))["impression_uid"].to_list()
    )
    FIT_ROWS = TRAINING_ROWS[TRAINING_ROWS["impression_uid"].isin(fit_uids)].reset_index(
        drop=True
    )
    ALL_ROWS = TRAINING_ROWS
    n_groups = int(TRAINING_ROWS["clicked"].sum())
    EPOCH_SAMPLER = EpochNegativeSampler(ALL_ROWS, CFG)


    def sample_epoch(epoch, *, fit_only):
        """Draw exposed negatives for one epoch and build popularity-aware arrays."""
        groups, _ = EPOCH_SAMPLER.draw(int(epoch), uids=fit_uids if fit_only else None)
        groups = (
            pl.from_pandas(groups)
            .join(profile_lookup, on="impression_uid", how="left")
            .with_row_index("group_row")
            .join(
                article_index.select(
                    pl.col("news_code"),
                    pl.col("article_row").alias("pos_row"),
                ),
                left_on="positive_news_code",
                right_on="news_code",
                how="left",
            )
        )
        if groups["profile_row"].null_count() or groups["pos_row"].null_count():
            raise ValueError("Sampled groups are missing training profiles or article rows")
        negative_long = (
            groups.select("group_row", "negative_news_codes")
            .explode("negative_news_codes", empty_as_null=True)
            .rename({"negative_news_codes": "negative_news_code"})
            .drop_nulls("negative_news_code")
            .with_columns(pl.int_range(pl.len()).over("group_row").alias("neg_pos"))
            .join(
                article_index.select(
                    pl.col("news_code").alias("negative_news_code"),
                    pl.col("article_row").alias("neg_row"),
                ),
                on="negative_news_code",
                how="left",
            )
            .sort(["group_row", "neg_pos"])
        )
        if negative_long["neg_row"].null_count():
            raise ValueError("Sampled negatives are missing article rows")
        n_epoch_groups = groups.height
        NEG_MAX = int(groups["negative_news_codes"].list.len().max() or 0)
        if n_epoch_groups and NEG_MAX == 0:
            raise ValueError("Sampled training rows contain no exposed negatives")
        NEG_MATRIX = np.zeros((n_epoch_groups, NEG_MAX), dtype=np.int64)
        group_rows = negative_long["group_row"].to_numpy().astype(np.int64)
        NEG_MATRIX[group_rows, negative_long["neg_pos"].to_numpy().astype(np.int64)] = (
            negative_long["neg_row"].to_numpy().astype(np.int64)
        )
        NEG_LEN = np.bincount(group_rows, minlength=n_epoch_groups).astype(np.int64)
        return (
            groups["profile_row"].to_numpy().astype(np.int32),
            groups["pos_row"].to_numpy().astype(np.int32),
            NEG_MATRIX,
            NEG_LEN,
            groups["hour"].to_numpy().astype(np.int64),
        )


    TUNE_NDCG_K = max_clicks_per_impression(TUNE_CANDIDATES)

    tune = tuning_slates(
        TUNE_CANDIDATES,
        cutoff_ns,
        article_index,
        PROFILE_INDEX_TRAIN,
        max_impressions=None,
        seed=SEED + 1,
    )
    TUNE_PROFILE_ROWS = tune["profile_row"].to_numpy().astype(np.int32)
    TUNE_ITEM_ROWS = tune["article_row"].to_numpy().astype(np.int32)
    TUNE_CLICKED = tune["clicked"].to_numpy().astype(np.float32)
    TUNE_UIDS = tune["impression_uid"].to_numpy()
    TUNE_HOURS = hour_index_of(
        tune["timestamp"].to_numpy().astype("datetime64[ns]").astype(np.int64)
    )
    if TUNE_HOURS.min() < 0 or TUNE_HOURS.max() >= N_HOURS:
        raise ValueError("Tuning rows fall outside the hourly popularity range")
    print(f"Training rows: {len(TRAINING_ROWS):,}")
    print(f"Training positives: {n_groups:,}")
    print(f"Chronological fit rows: {len(FIT_ROWS):,} / {len(ALL_ROWS):,}")
    print(f"Sampler negatives per positive: up to {CFG.negative_count}")
    print("Tuning cutoff:", np.datetime64(cutoff_ns, "ns"))
    print(f"Tuning impressions: {tune['impression_uid'].n_unique():,}")
    print(f"Tuning pairs: {tune.height:,}")


Training rows: 3,389,342
Training positives: 141,558
Chronological fit rows: 3,046,331 / 3,389,342
Sampler negatives per positive: up to 4
Tuning cutoff: 2019-11-12T13:50:10.000000000
Tuning impressions: 9,507
Tuning pairs: 343,011


## 3. Model

`TwoTower` has two independent MLPs with the same hidden shape. The request tower maps the 166-d pooled history profile through `166 -> 256` with GELU and dropout `0.1`, then `256 -> 256`, followed by L2 normalisation. The item tower maps a 168-d article vector through `168 -> 256`, then `256 -> 256` and L2 normalisation; the input concatenates intrinsic content with `log1p` prior-hour exposures and `log1p` prior-hour clicks. The two-tower score is a learned-temperature dot product, `exp(logit_scale) * (u . v)`, with `logit_scale` initialised to `log(10)`.

At serving only, a frozen additive popularity term is added:

```text
final_score = exp(logit_scale) * (u . v)
              + exposure_weight * log1p(prior_hour_exposures)
              + click_weight    * log1p(prior_hour_clicks)
```

The additive weights are tuned on the chronological train tuning slates and are never part of the training objective.

Architecture:

```text
request profile (166) --> user tower: 166 -> 256 (GELU, dropout 0.1) -> 256 -> L2 norm -> u
article input (168)   --> item tower: 168 -> 256 (GELU, dropout 0.1) -> 256 -> L2 norm -> v

training score = exp(logit_scale) * (u . v)      logit_scale initialised to log(10)
serving score  = training score + alpha * prior_exposures + beta * prior_clicks

BPR objective (one positive vs up to 4 exposed negatives per step):

s_pos = training_score(u, v_clicked)
s_neg = training_score(u, v_shown_not_clicked)
loss  = -mean( log sigmoid(s_pos - s_neg) )
```


In [52]:
class TwoTower(nn.Module):
    def __init__(
        self,
        user_input_dim=FEATURE_DIM,
        item_input_dim=ITEM_FEATURE_DIM,
        hidden_dim=HIDDEN_DIM,
        embed_dim=EMBED_DIM,
        dropout=DROPOUT,
        scale=10.0,
    ):
        super().__init__()
        self.user_tower = self._tower(user_input_dim, hidden_dim, embed_dim, dropout)
        self.item_tower = self._tower(item_input_dim, hidden_dim, embed_dim, dropout)
        self.logit_scale = nn.Parameter(
            torch.tensor(float(np.log(scale)), dtype=torch.float32)
        )

    @staticmethod
    def _tower(input_dim, hidden_dim, embed_dim, dropout):
        return nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim),
        )

    def encode_users(self, values):
        return F.normalize(self.user_tower(values), dim=-1)

    def encode_items(self, values):
        return F.normalize(self.item_tower(values), dim=-1)

    def forward(self, user_values, item_values):
        return torch.exp(self.logit_scale) * (
            self.encode_users(user_values) * self.encode_items(item_values)
        ).sum(dim=-1)

## 4. Training

Every epoch rebuilds the negative matrix from `EpochNegativeSampler`, then optimises BPR on `positive_score - negative_score` with AdamW; a BCE variant is available through the objective switch. Training item inputs include each slate's prior-hour exposure and click counts, while the additive popularity term is excluded from the loss. After restoring the best tuning epoch, the notebook grid-searches exposure and click weights on the same chronological tuning slates, writes `serving_weight_search.csv`, freezes the best pair, and refits a fresh model for that many epochs on all training rows. The checkpoint and training log are written under `artifacts/two_tower_content_popularity_<loss>/`.


In [53]:
if TRAINING_REQUIRED:
    def tune_scores(model):
        model.eval()
        scores = np.empty(len(TUNE_CLICKED), dtype=np.float32)
        with torch.no_grad():
            for start in range(0, len(TUNE_CLICKED), 200_000):
                end = min(start + 200_000, len(TUNE_CLICKED))
                users = torch.as_tensor(
                    TRAIN_USER_FEATURES[TUNE_PROFILE_ROWS[start:end]],
                    dtype=torch.float32,
                    device=DEVICE,
                )
                items = torch.as_tensor(
                    item_features(TUNE_ITEM_ROWS[start:end], TUNE_HOURS[start:end]),
                    device=DEVICE,
                )
                scores[start:end] = model(users, items).cpu().numpy()
        return scores


    def tune_ndcg(model):
        _, ndcg = mean_slate_metrics(
            TUNE_UIDS, TUNE_CLICKED, tune_scores(model), TUNE_NDCG_K
        )
        return ndcg


    def select_serving_weights(model):
        base_scores = tune_scores(model)
        tune_exposures, tune_clicks = prior_hour_counts(TUNE_HOURS, TUNE_ITEM_ROWS)
        search = []
        best = None
        for exposure_weight in SERVING_WEIGHT_GRID:
            for click_weight in SERVING_WEIGHT_GRID:
                scores = (
                    base_scores
                    + exposure_weight * tune_exposures
                    + click_weight * tune_clicks
                )
                _, ndcg = mean_slate_metrics(TUNE_UIDS, TUNE_CLICKED, scores, TUNE_NDCG_K)
                row = {
                    "exposure_weight": exposure_weight,
                    "click_weight": click_weight,
                    "tune_ndcg": ndcg,
                    "ndcg_k": TUNE_NDCG_K,
                }
                search.append(row)
                if best is None or ndcg > best["tune_ndcg"] + 1e-9:
                    best = row
        return (
            {
                "exposures": float(best["exposure_weight"]),
                "clicks": float(best["click_weight"]),
            },
            pl.DataFrame(search),
        )


In [54]:
if TRAINING_REQUIRED:
    def train_epoch(
        model,
        optimizer,
        profile_rows,
        pos_item_rows,
        neg_matrix,
        neg_len,
        group_hours,
        rng,
        objective,
    ):
        model.train()
        order = rng.permutation(len(profile_rows))
        total = 0.0
        for start in range(0, len(order), BATCH_SIZE):
            batch = order[start : start + BATCH_SIZE]
            users = torch.as_tensor(
                TRAIN_USER_FEATURES[profile_rows[batch]],
                dtype=torch.float32,
                device=DEVICE,
            )
            positives = torch.as_tensor(
                item_features(pos_item_rows[batch], group_hours[batch]),
                device=DEVICE,
            )
            neg_rows = neg_matrix[batch]
            neg_lens = torch.as_tensor(neg_len[batch], device=DEVICE)
            neg_mask = (
                torch.arange(neg_rows.shape[1], device=DEVICE)[None, :] < neg_lens[:, None]
            )
            users_expanded = (
                users[:, None, :]
                .expand(-1, neg_rows.shape[1], -1)
                .reshape(-1, users.shape[1])
            )
            negative_inputs = torch.as_tensor(
                item_features(
                    neg_rows.reshape(-1),
                    np.repeat(group_hours[batch], neg_rows.shape[1]),
                ),
                device=DEVICE,
            )
            optimizer.zero_grad(set_to_none=True)
            positive_scores = model(users, positives)
            negative_scores = model(users_expanded, negative_inputs).reshape(
                len(batch), neg_rows.shape[1]
            )
            if objective == "bpr":
                pair_loss = F.logsigmoid(positive_scores[:, None] - negative_scores)
                loss = -(pair_loss * neg_mask).sum() / neg_mask.sum()
            else:
                scores = torch.cat([positive_scores, negative_scores[neg_mask]])
                labels = torch.cat(
                    [
                        torch.ones_like(positive_scores),
                        torch.zeros_like(negative_scores[neg_mask]),
                    ]
                )
                loss = F.binary_cross_entropy_with_logits(scores, labels)
            loss.backward()
            optimizer.step()
            total += float(loss.item()) * len(batch)
        return total / len(order)


In [55]:
def build_model():
    return TwoTower().to(DEVICE)

In [ ]:
TRAINED_LOSSES = {}
for objective in RUN_LOSSES:
    config = LOSS_CONFIGS[objective]
    artifact_dir = config["artifact_dir"]
    print(f"[{objective}] Training {config['model_name']}")

    model_path = artifact_dir / "model.pt"
    expected_config = {
        "user_input_dim": FEATURE_DIM,
        "item_input_dim": ITEM_FEATURE_DIM,
        "hidden_dim": HIDDEN_DIM,
        "embed_dim": EMBED_DIM,
        "dropout": DROPOUT,
        "popularity_features": list(POPULARITY_FEATURES),
        "objective": objective,
    }
    if REUSE_ARTIFACTS and model_path.exists():
        checkpoint = torch.load(model_path, map_location=DEVICE, weights_only=False)
        saved_config = checkpoint.get("config", {})
        if all(saved_config.get(key) == value for key, value in expected_config.items()):
            serving_weights = saved_config.get("serving_popularity_weights")
            if serving_weights is None:
                raise ValueError(
                    "Checkpoint is missing its serving popularity weights"
                )
            model = build_model()
            model.load_state_dict(checkpoint["state_dict"])
            model.eval()
            TRAINED_LOSSES[objective] = {
                "model": model,
                "config": config,
                "serving_weights": serving_weights,
            }
            print(f"[{objective}] Loaded model: {model_path}")
            continue
        print(f"[{objective}] Checkpoint config mismatch; retraining")

    model = build_model()
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    rng = np.random.default_rng(SEED + 1)
    training_log = []
    stopper = EarlyStopper(patience=PATIENCE)

    for epoch in range(1, NUM_EPOCHS + 1):
        started = time.time()
        profile_rows, pos_rows, neg_matrix, neg_len, group_hours = sample_epoch(
            epoch - 1, fit_only=True
        )
        loss = train_epoch(
            model,
            optimizer,
            profile_rows,
            pos_rows,
            neg_matrix,
            neg_len,
            group_hours,
            rng,
            objective,
        )
        ndcg = tune_ndcg(model)
        elapsed = time.time() - started
        training_log.append(
            {
                "phase": "tune",
                "epoch": epoch,
                "train_loss": loss,
                "tune_ndcg": ndcg,
                "ndcg_k": TUNE_NDCG_K,
                "seconds": elapsed,
            }
        )
        print(
            f"epoch {epoch:02d} loss={loss:.6f} "
            f"tune_ndcg@{TUNE_NDCG_K}={ndcg:.5f} ({elapsed:.1f}s)"
        )
        if stopper.step(ndcg, model, epoch):
            print("Early stopping")
            break

    stopper.restore(model)
    print(
        f"Best epoch: {stopper.best_epoch} "
        f"(tune_ndcg@{TUNE_NDCG_K}={stopper.best_score:.5f})"
    )

    serving_weights, weight_search = select_serving_weights(model)
    weight_search.write_csv(artifact_dir / "serving_weight_search.csv")
    print(
        "Serving popularity weights:",
        f"exposures={serving_weights['exposures']:.4f}",
        f"clicks={serving_weights['clicks']:.4f}",
    )

    final_model = build_model()
    optimizer = torch.optim.AdamW(
        final_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    rng = np.random.default_rng(SEED + 2)

    for epoch in range(1, stopper.best_epoch + 1):
        started = time.time()
        profile_rows, pos_rows, neg_matrix, neg_len, group_hours = sample_epoch(
            epoch - 1, fit_only=False
        )
        loss = train_epoch(
            final_model,
            optimizer,
            profile_rows,
            pos_rows,
            neg_matrix,
            neg_len,
            group_hours,
            rng,
            objective,
        )
        elapsed = time.time() - started
        training_log.append(
            {
                "phase": "refit",
                "epoch": epoch,
                "train_loss": loss,
                "tune_ndcg": float("nan"),
                "ndcg_k": TUNE_NDCG_K,
                "seconds": elapsed,
            }
        )
        print(f"refit epoch {epoch:02d} loss={loss:.6f} ({elapsed:.1f}s)")

    torch.save(
        {
            "state_dict": final_model.state_dict(),
            "config": {
                "user_input_dim": FEATURE_DIM,
                "item_input_dim": ITEM_FEATURE_DIM,
                "hidden_dim": HIDDEN_DIM,
                "embed_dim": EMBED_DIM,
                "dropout": DROPOUT,
                "item_features": (
                    "text64 + entity100 + missing-vector flags + "
                    "log1p prior-hour exposures + log1p prior-hour clicks"
                ),
                "popularity_features": list(POPULARITY_FEATURES),
                "serving_popularity_weights": serving_weights,
                "serving_weight_grid": list(SERVING_WEIGHT_GRID),
                "objective": objective,
                "epochs": stopper.best_epoch,
                "tune_ndcg": stopper.best_score,
                "ndcg_k": TUNE_NDCG_K,
                "batch_size": BATCH_SIZE,
                "learning_rate": LEARNING_RATE,
                "weight_decay": WEIGHT_DECAY,
                "seed": SEED,
                "negative_sampling": "per-epoch exposed negatives",
                "negative_count": int(CFG.negative_count),
                "sampler_epoch_base": 0,
                "training_data": "labels_train full training pool",
            },
        },
        artifact_dir / "model.pt",
    )
    pl.DataFrame(training_log).write_csv(artifact_dir / "training_log.csv")
    print("Saved model:", artifact_dir / "model.pt")

    TRAINED_LOSSES[objective] = {
        "model": final_model,
        "config": config,
        "serving_weights": serving_weights,
    }

## 5. Generate the standard candidate table

`select_top_k` scores each request against articles available by its timestamp. User embeddings are encoded once; item embeddings are encoded per clock hour because they now include that hour's prior-hour popularity features. The final score adds the frozen exposure and click weights, then removes the request's history, ranks by score descending, previous-hour click popularity descending, then ascending `news_code`, and backfills short lists from popularity. Requests with no content profile receive the hourly popularity order directly. Rows are streamed through `CandidateWriter` in timestamp order to `artifacts/two_tower_content_popularity_<loss>/recommendations_<split>_top500.parquet`.


In [57]:
def popularity_select(order, n_eligible, excluded, k):
    excluded_set = set(map(int, excluded))
    selected = []
    for position in order:
        position = int(position)
        if position >= n_eligible or position in excluded_set:
            continue
        selected.append(position)
        if len(selected) == k:
            return np.asarray(selected, dtype=np.int64)
    raise ValueError(f"only {len(selected)} eligible articles; cannot select {k}")


def popularity_fill(available, popularity, news_code, count):
    pool = np.flatnonzero(available)
    order = np.lexsort((news_code[pool], -popularity[pool]))
    return pool[order[:count]]


def select_top_k(scores, popularity, news_code, k, excluded, fallback_order):
    if fallback_order is not None:
        return popularity_select(fallback_order, len(scores), excluded, k)

    available = np.ones(len(scores), dtype=bool)
    if len(excluded):
        available[excluded] = False
        scores = scores.copy()
        scores[excluded] = -np.inf

    finite = np.isfinite(scores)
    n_finite = int(np.count_nonzero(finite))
    if n_finite == 0:
        return popularity_fill(available, popularity, news_code, k)

    take = min(k, n_finite)
    if take < n_finite:
        threshold = scores[np.argpartition(-scores, take - 1)[:take]].min()
    else:
        threshold = scores[finite].min()

    above = np.flatnonzero(scores > threshold)
    need = take - above.size
    ties = np.flatnonzero(scores == threshold)
    if need > 0:
        if ties.size > need:
            ties = ties[np.lexsort((news_code[ties], -popularity[ties]))[:need]]
        chosen = np.r_[above, ties]
    else:
        chosen = above

    order = np.lexsort((news_code[chosen], -popularity[chosen], -scores[chosen]))
    selected = chosen[order].astype(np.int64)
    if selected.size < k:
        remaining = available.copy()
        remaining[selected] = False
        selected = np.r_[
            selected,
            popularity_fill(remaining, popularity, news_code, k - selected.size),
        ]
    return selected

In [58]:
def encode_batches(model, features, encode, batch=100_000):
    out = np.empty((len(features), EMBED_DIM), dtype=np.float32)
    model.eval()
    with torch.no_grad():
        for start in range(0, len(features), batch):
            values = torch.as_tensor(
                features[start : start + batch],
                dtype=torch.float32,
                device=DEVICE,
            )
            out[start : start + len(values)] = encode(values).cpu().numpy()
    return out

In [59]:
for objective in RUN_LOSSES:
    state = TRAINED_LOSSES[objective]
    config = state["config"]
    model = state["model"]
    serving_weights = state["serving_weights"]
    print(f"[{objective}] Scoring {config['model_name']}")

    target_profiles = load_profiles(TARGET_SPLIT)
    TARGET_USER_FEATURES, _ = load_profile_cache(TARGET_SPLIT)

    model.eval()
    user_emb = encode_batches(model, TARGET_USER_FEATURES, model.encode_users)

    has_profile = (TARGET_USER_FEATURES[:, -2] > 0) | (TARGET_USER_FEATURES[:, -1] > 0)

    impression_uids = target_profiles["impression_uid"].to_numpy()
    timestamps = target_profiles["timestamp"].to_numpy().astype("datetime64[ns]")
    target_hours = hour_index_of(timestamps.astype(np.int64))
    history_rows = target_profiles["full_history_rows"].to_list()
    starts = np.flatnonzero(np.r_[True, timestamps[1:] != timestamps[:-1]])
    ends = np.r_[starts[1:], len(timestamps)]
    batches = list(zip(starts, ends))
    print(f"Scoring requests: {len(timestamps):,}")

    cutoff_values = np.arange(1, CANDIDATE_CAP + 1, dtype=np.int16)
    writer = CandidateWriter(config["recommendations_path"])
    current_hour = None
    popularity_exposures = None
    popularity_clicks = None
    item_emb = None
    fallback_order = None
    started = time.time()
    try:
        for batch_no, (start, end) in enumerate(batches, 1):
            timestamp = timestamps[start]
            batch_hour = int(target_hours[start])
            if batch_hour != current_hour:
                popularity_exposures, popularity_clicks = prior_hour_counts(
                    batch_hour, ARTICLE_ROWS_SORTED
                )
                item_emb = encode_batches(
                    model,
                    item_features(ARTICLE_ROWS_SORTED, batch_hour),
                    model.encode_items,
                )
                fallback_order = np.lexsort(
                    (NEWS_SORTED, -popularity_exposures, -popularity_clicks)
                )
                current_hour = batch_hour

            n_eligible = int(np.searchsorted(AVAILABLE_SORTED, timestamp, side="right"))
            scores = user_emb[start:end] @ item_emb[:n_eligible].T
            scores = (
                scores
                + serving_weights["exposures"] * popularity_exposures[:n_eligible]
                + serving_weights["clicks"] * popularity_clicks[:n_eligible]
            )
            popularity_batch = popularity_clicks[:n_eligible]
            news_batch = NEWS_SORTED[:n_eligible]
            selected_batch = np.empty((end - start, CANDIDATE_CAP), dtype=np.int64)
            for local, profile_row in enumerate(range(start, end)):
                rows = history_rows[profile_row]
                excluded = (
                    POS_IN_SORTED[np.asarray(rows, dtype=np.int64)]
                    if rows
                    else np.empty(0, dtype=np.int64)
                )
                fallback = None if has_profile[profile_row] else fallback_order
                selected_batch[local] = select_top_k(
                    scores[local],
                    popularity_batch,
                    news_batch,
                    CANDIDATE_CAP,
                    excluded,
                    fallback,
                )

            writer.add(
                np.repeat(impression_uids[start:end], CANDIDATE_CAP),
                np.tile(cutoff_values, end - start),
                news_batch[selected_batch.reshape(-1)],
            )
            if batch_no % 1000 == 0 or batch_no == len(batches):
                print(
                    f"Scored {batch_no:,}/{len(batches):,} timestamp batches; "
                    f"{writer.rows:,} rows",
                    flush=True,
                )
    finally:
        writer.close()

    elapsed = time.time() - started
    print(f"Wrote {len(timestamps) * CANDIDATE_CAP:,} candidate rows in {elapsed:.1f}s")
    print("Candidates:", config["recommendations_path"])
    state["recommendations_path"] = config["recommendations_path"]

[bpr] Scoring content_two_tower_popularity_mlp_bpr
Scoring requests: 31,624
Scored 1,000/24,580 timestamp batches; 523,500 rows
Scored 2,000/24,580 timestamp batches; 1,099,000 rows
Scored 3,000/24,580 timestamp batches; 1,718,000 rows
Scored 4,000/24,580 timestamp batches; 2,367,000 rows
Scored 5,000/24,580 timestamp batches; 3,048,000 rows
Scored 6,000/24,580 timestamp batches; 3,734,000 rows
Scored 7,000/24,580 timestamp batches; 4,404,000 rows
Scored 8,000/24,580 timestamp batches; 5,095,500 rows
Scored 9,000/24,580 timestamp batches; 5,771,000 rows
Scored 10,000/24,580 timestamp batches; 6,465,500 rows
Scored 11,000/24,580 timestamp batches; 7,188,000 rows
Scored 12,000/24,580 timestamp batches; 7,901,000 rows
Scored 13,000/24,580 timestamp batches; 8,620,500 rows
Scored 14,000/24,580 timestamp batches; 9,331,000 rows
Scored 15,000/24,580 timestamp batches; 10,035,500 rows
Scored 16,000/24,580 timestamp batches; 10,730,000 rows
Scored 17,000/24,580 timestamp batches; 11,417,000 ro

## 6. Evaluation

Uses the shared `CandidateEvaluator` and writes `results/candidate_generation_<model>_<split>.csv`. It reports recall (pooled and impression-mean), hit rate, long-tail recall, cold-article recall, novelty (bits), catalogue coverage and personalisation at each M.


In [60]:
from cg_evaluation import CandidateEvaluator

for loss in RUN_LOSSES:
    state = TRAINED_LOSSES[loss]
    config = state["config"]
    evaluator = CandidateEvaluator(root=DATA_ROOT, split=TARGET_SPLIT)
    evaluation = evaluator.evaluate_to_csv(
        state["recommendations_path"],
        model_name=config["model_name"],
        m_values=M_VALUES,
        output_dir=RESULTS,
    )
    print(f"[{loss}]")
    print(evaluation.sort("m"))
    print("Results:", config["result_path"])

[bpr]
shape: (32, 7)
┌───────────────────┬────────────┬─────┬───────────────────┬───────────┬─────────────┬─────────────┐
│ model             ┆ split      ┆ m   ┆ metric            ┆ value     ┆ numerator   ┆ denominator │
│ ---               ┆ ---        ┆ --- ┆ ---               ┆ ---       ┆ ---         ┆ ---         │
│ str               ┆ str        ┆ i64 ┆ str               ┆ f64       ┆ f64         ┆ f64         │
╞═══════════════════╪════════════╪═════╪═══════════════════╪═══════════╪═════════════╪═════════════╡
│ content_two_tower ┆ validation ┆ 50  ┆ catalogue_coverag ┆ 0.111637  ┆ 5307.0      ┆ 47538.0     │
│ _popularity_m…    ┆            ┆     ┆ e                 ┆           ┆             ┆             │
│ content_two_tower ┆ validation ┆ 50  ┆ cold_article_reca ┆ 0.0       ┆ 0.0         ┆ 342.0       │
│ _popularity_m…    ┆            ┆     ┆ ll                ┆           ┆             ┆             │
│ content_two_tower ┆ validation ┆ 50  ┆ hit_rate          ┆ 0.059385 

## Notes and limitations

- The user representation is specific to each request. Repeat users do not share a single latest profile; earlier requests use only earlier feedback.
- The item tower now consumes causal prior-hour exposure and click counts in addition to intrinsic content. The same values feed the serving-only additive popularity term.
- The additive exposure and click weights are tuned on the chronological train tuning slates and are never optimised in the training loss.
- Excluding the causal history removes repeat-click opportunities, matching the content baseline, so recall does not measure repeat clicks.
- Popularity is strictly prior-hour: a request in hour `h` reads only completed hours before `h`, so no target-period or same-hour information enters the score or the item embedding.
- Item embeddings are no longer static because they depend on the request hour; they are encoded once per unique serving hour.
- Up to four exposed negatives are redrawn from the same slate each epoch, so the model inherits the logging system's exposure and position bias.
- Training uses a chronological 90% tuning slice, early stopping and a final refit on all training rows.
- The score is a ranking score. Grouped exposed negatives make the training positive rate much higher than held-out CTR, so the output is not calibrated.
- `available_from` is the project-wide observable-availability proxy, not a verified publication timestamp.
